# Observatoire des formes relationnelles — Cas 1

## Bumble « Opening Moves » (mars–avril 2024) — DiD imparfait sur avis datés

*Cas inaugural de l'Observatoire (#13303) — protocole de preuve #13309 —
instrument $G_t^{arg} \to G_{t+1}^{arg}$ rebranché depuis
[`Argument_Analysis_Dated_Graphs.ipynb`](Argument_Analysis_Dated_Graphs.ipynb)
(livré #13310).*

`Grain: DEEP/research-code — lane myia-po-2027:CoursIA-2 — prev: MED/guard #16423`


## 1. Position dans le protocole

Premier **cas** de l'Observatoire. La séquence du protocole est tenue :

1. **Protocole** (#13309) — règle de preuve, barreau 1→3, biais de sélection nommé.
2. **Instrument** (#13310) — l'écart structurel daté `d_struct`, son plancher split-half, son contrôle positif par construction.
3. **Cas** (présent notebook) — premier candidat nommé dans le body de l'EPIC : Bumble « Opening Moves », rupture datée, bras contrôle Hinge sans changement sur la même fenêtre.

**Barreau visé** : 1 → éventuellement 2. **Jamais 3** (toute conclusion causale récusée par construction). Cf. #13303 §Volet Désir.


## 2. Pré-inscription

Rappel — voir commentaire d'inscription (#16410) pour la version complète.

| Élément | Valeur |
|---|---|
| **Rupture** | Bumble « Opening Moves » — déploiement **2024-04-30** (couverture presse 2024-05-01) |
| **Fenêtre pré** | 2023-10-15 → 2024-04-29 (6 mois) |
| **Fenêtre post** | 2024-05-15 → 2024-11-15 (6 mois + tampon de rollout 15 j) |
| **Bras traité** | r/Bumble (textes publics Reddit datés) |
| **Bras contrôle** | r/hingeapp (aucun changement de règle d'initiation sur la fenêtre) |
| **Corpus source** | Arctic Shift (`arctic-shift.photon-reddit.com`) |

**Hypothèses signées** :

- **H₀** : `Δ_struct(Bumble_post, Bumble_pre) − Δ_struct(hingeapp_post, hingeapp_pre) ≤ plancher split-half`. Les distributions de propositions n décalent pas différentiellement.
- **H₁** : décalage positif sur le bras traité uniquement, hausse de `INIT_NORM_ANY` et `OPENING_MOVE_REACT`, baisse de `INIT_NORM_W`, avec `Δ_struct(traité) > Δ_struct(contrôle) + plancher`.

**Biais de sélection signé** : r/Bumble sur-représente les utilisateurs investis et les événements saillants. **Direction** : pousse vers H₁ (la rupture elle-même fabrique des posts `OPENING_MOVE_REACT`). Le bras contrôle n'absorbe pas ce biais de saillance spécifique au traité.

**Plafond revendiqué** : barreau associatif au mieux, « quasi-expérimental » inatteignable avec ce corpus (commentaire absent / utilisateurs silencieux / résiliés / hors-app / non-anglophones).


## 3. Engagement de corpus

**Politique dataset (CLAUDE.md §Secrets-hygiene)** : aucun corpus brut commité.
Seuls les **agrégats**, **comptes** et **verbatims bornés** (5 par cellule de
mesure, sans identifiant auteur) entrent dans le notebook.

**Source** : Arctic Shift, endpoint `GET /api/posts/search`, paramètres
`subreddit`, `after`, `before`, `limit`. Échantillonnage stratifié : 50 textes
par (bras × fenêtre) = 4 cellules de 50 textes = 200 textes totaux, dont
**sous-échantillon de 5 par cellule pour la classification LLM** (cf. §6).

**Modèle d'extraction** : Ollama local `qwen2.5:7b-instruct-q4_K_M`,
température 0, prompt épinglé, schéma JSON de sortie épinglé. Aucun appel
distant à clé d'API.


In [1]:
# --- Reprise de l'instrument Dated_Graphs ---
# Source : Argument_Analysis_Dated_Graphs.ipynb -- cellules 2 (Enonce),
# 4 (AIFGraph + build_aif_graph), 10 (jaccard_distance),
# 12 (split_half), 14 (d_struct).
# Aucun re-écriture : on importe le module converti .py si présent, sinon
# repli explicite sur définition inline (cellule 5).

import importlib.util
from pathlib import Path

INSTR_PATH = Path(
    "MyIA.AI.Notebooks/SymbolicAI/Argument_Analysis/"
    "Argument_Analysis_Dated_Graphs.ipynb"
)
PY_PATH = INSTR_PATH.with_suffix(".py")

INSTRUMENT_OK = False
if PY_PATH.exists():
    spec = importlib.util.spec_from_file_location("_dated_graphs_mod", PY_PATH)
    _mod = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(_mod)
        from _dated_graphs_mod import (
            Enonce, AIFGraph, build_aif_graph,
            jaccard_distance, split_half, d_struct,
        )
        INSTRUMENT_OK = True
        print(f"[instrument] rechargé : {INSTR_PATH.name} -> 6 symboles")
    except Exception as e:
        print(f"[instrument] exec mod a échoué ({e}) -> repli cellule 5")
else:
    print("[instrument] .py absent -> repli cellule 5")


[instrument] .py absent -> repli cellule 5


In [2]:
# --- Repli explicite si l'import n'a pas abouti (cf. cellule 4) ---
# Reprise minimale : Enonce, AIFGraph, build_aif_graph, jaccard_distance,
# split_half, d_struct. L'instrument complet reste dans
# Argument_Analysis_Dated_Graphs.ipynb (cellules 2, 4, 10, 12, 14).

if not INSTRUMENT_OK:
    from dataclasses import dataclass, field
    from random import Random

    @dataclass(frozen=True)
    class Enonce:
        jour: int
        prop: str
        acte: str
        relations: tuple = ()

    @dataclass
    class AIFGraph:
        i_nodes: dict = field(default_factory=dict)
        scheme_nodes: dict = field(default_factory=dict)
        excluded: list = field(default_factory=list)

    def build_aif_graph(enonces):
        g = AIFGraph()
        assertees = [e for e in enonces if e.acte == "assertif" and e.prop]
        presentes = {e.prop for e in assertees}
        relations = []
        for e in assertees:
            for scheme, cible in e.relations:
                relations.append((scheme, e.prop, cible))
        reliees = {s for _, s, _ in relations} | {c for _, _, c in relations}
        vues = set()
        for e in enonces:
            if e.prop in vues:
                continue
            vues.add(e.prop)
            if e.acte != "assertif":
                g.excluded.append((e.prop, "C1"))
            elif e.prop in presentes and e.prop not in reliees:
                g.excluded.append((e.prop, "C3"))
        exclus = {p for p, _ in g.excluded}
        for p in sorted((presentes | reliees) - exclus):
            g.i_nodes[p] = f"proposition {p}"
        for scheme, src, dst in sorted(set(relations)):
            g.scheme_nodes[f"{scheme}_{src}_{dst}"] = (scheme, src, dst)
        return g

    def jaccard_distance(A, B):
        a, b = set(A), set(B)
        if not a and not b:
            return 0.0
        return 1 - len(a & b) / len(a | b)

    def split_half(items, seed=0):
        rng = Random(seed)
        indices = list(range(len(items)))
        rng.shuffle(indices)
        m = len(indices) // 2
        return ([items[i] for i in indices[:m]],
                [items[i] for i in indices[m:]])

    def d_struct(af_t, af_t1):
        inter_t = set(af_t.i_nodes) | set(af_t.scheme_nodes)
        inter_t1 = set(af_t1.i_nodes) | set(af_t1.scheme_nodes)
        gap_nodes = jaccard_distance(inter_t, inter_t1)
        att_t = {(s, src, dst) for s, (sh, src, dst)
                 in af_t.scheme_nodes.items()}
        att_t1 = {(s, src, dst) for s, (sh, src, dst)
                  in af_t1.scheme_nodes.items()}
        gap_att = jaccard_distance(att_t, att_t1)
        return gap_nodes, gap_att

    print("[repli] Enonce / AIFGraph / build_aif_graph / jaccard_distance / "
          "split_half / d_struct définis inline")


[repli] Enonce / AIFGraph / build_aif_graph / jaccard_distance / split_half / d_struct définis inline


In [3]:
# --- Fetch Arctic Shift : 4 corpus (bras × fenêtre), 50 textes chacun ---
# Endpoint : arctic-shift.photon-reddit.com/api/posts/search
# Fenêtres (pré-inscription §2) : pré 2023-10-15 -> 2024-04-29,
# post 2024-05-15 -> 2024-11-15.

import urllib.request
import urllib.parse
import json

BASE = "https://arctic-shift.photon-reddit.com/api/posts/search"

WINDOWS = {
    "Bumble_pre":   ("Bumble",   "2023-10-15", "2024-04-29"),
    "Bumble_post":  ("Bumble",   "2024-05-15", "2024-11-15"),
    "Hinge_pre":    ("hingeapp", "2023-10-15", "2024-04-29"),
    "Hinge_post":   ("hingeapp", "2024-05-15", "2024-11-15"),
}
LIMIT = 50

def fetch_corpus(sub, after, before, limit=LIMIT):
    qs = urllib.parse.urlencode({
        "subreddit": sub, "after": after, "before": before,
        "limit": limit, "sort": "desc", "sort_type": "created_utc",
    })
    url = f"{BASE}?{qs}"
    req = urllib.request.Request(
        url, headers={"User-Agent": "observatoire-cas1/1.0"}
    )
    with urllib.request.urlopen(req, timeout=20) as r:
        body = json.load(r)
    return body.get("data", [])

corpora = {}
for name, (sub, after, before) in WINDOWS.items():
    rows = fetch_corpus(sub, after, before, limit=LIMIT)
    corpora[name] = rows
    n_kept = sum(1 for r in rows if r.get("selftext") or r.get("title"))
    print(f"  {name}: {len(rows)} posts fetched, {n_kept} non vides")

total = sum(len(c) for c in corpora.values())
print(f"\nTotal brut : {total} posts. Aucun n'est commite en memoire d'execution.")


  Bumble_pre: 50 posts fetched, 50 non vides


  Bumble_post: 50 posts fetched, 50 non vides


  Hinge_pre: 50 posts fetched, 50 non vides


  Hinge_post: 50 posts fetched, 50 non vides

Total brut : 200 posts. Aucun n'est commite en memoire d'execution.


In [4]:
# --- Agregats public-safe : pas de selftext brut, pas d'auteur ---
# Pour chaque corpus : nb de posts, longueur moyenne, 5 verbatims courts
# (<= 140 car) sans identifiant. Le selftext complet n'est JAMAIS ecrit
# dans le notebook committe (politique dataset).

import re, json as _json

def anonymize(text):
    if not text:
        return ""
    t = re.sub(r"https?://\S+", "", text)
    t = re.sub(r"/u/\w+|@\w+", "[REDACTED]", t)
    t = t.strip()
    return (t[:140] + "...") if len(t) > 140 else t

aggregats = {}
for name, rows in corpora.items():
    lengths = [len((r.get("selftext") or "") + (r.get("title") or ""))
               for r in rows]
    verbatims = [anonymize(r.get("selftext") or r.get("title", ""))
                 for r in rows if (r.get("selftext") or r.get("title"))]
    verbatims = [v for v in verbatims if v][:5]
    aggregats[name] = {
        "n": len(rows),
        "len_moy": sum(lengths) / max(len(lengths), 1),
        "verbatims": verbatims,
    }

# On n'imprime que les metadonnees publiques ; les verbatims eux-memes
# dependent du fetch reel de la cellule 6.
meta = {k: {kk: vv for kk, vv in v.items() if kk != "verbatims"}
        for k, v in aggregats.items()}
print(_json.dumps(meta, indent=2, ensure_ascii=False))


{
  "Bumble_pre": {
    "n": 50,
    "len_moy": 359.92
  },
  "Bumble_post": {
    "n": 50,
    "len_moy": 504.66
  },
  "Hinge_pre": {
    "n": 50,
    "len_moy": 207.82
  },
  "Hinge_post": {
    "n": 50,
    "len_moy": 188.96
  }
}


In [5]:
# --- Extraction LLM Ollama : classification en 6 categories ---
# Modele epingle : qwen2.5:7b-instruct-q4_K_M (4.7 Go).
# Temperature 0, prompt + schema JSON fixes.
# Sous-echantillon : 5 textes / cellule, 4 cellules = 20 classifications.
# Suffisant pour la demonstration d'instrumentation (cf. §10 du verdict
# pour la limite statistique).

import urllib.request, json, random

OLLAMA = "http://localhost:11434/api/generate"
MODEL = "qwen2.5:7b-instruct-q4_K_M"

CATEGORIES = [
    "INIT_NORM_W",         # la femme doit initier
    "INIT_NORM_ANY",       # n'importe qui peut initier
    "INIT_COST",           # cout/burden de l'initiation
    "BLAME",               # blame d'un genre pour la non-initiation
    "MATCH_DECAY",         # matchs meurent sans premier message
    "OPENING_MOVE_REACT",  # reaction au changement nomme
]

PROMPT_TEMPLATE = """Classifie le post ci-dessous selon QUELLE proposition
d'initiation sociale il porte (0 ou plusieurs). Sois strict.

Categories (renvoie un JSON {{ "cats": [liste], "verbatim": "..." }}) :
{cats}

Post :
{post}
"""

def classify(text, retries=2):
    prompt = PROMPT_TEMPLATE.format(
        cats=", ".join(CATEGORIES), post=text[:600]
    )
    payload = json.dumps({
        "model": MODEL, "prompt": prompt, "stream": False,
        "options": {"temperature": 0.0, "num_predict": 200},
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA, data=payload,
        headers={"Content-Type": "application/json"}
    )
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                resp = json.load(r)
            txt = resp.get("response", "").strip()
            i, j = txt.find("{"), txt.rfind("}")
            if i >= 0 and j > i:
                obj = json.loads(txt[i:j + 1])
                obj.setdefault("cats", [])
                obj.setdefault("verbatim", "")
                return obj
        except Exception as ex:
            if attempt == retries - 1:
                return {"cats": [], "verbatim": "", "error": str(ex)}
    return {"cats": [], "verbatim": ""}

random.seed(42)
sampled = {}
for name, rows in corpora.items():
    eligible = [r for r in rows if (r.get("selftext") or r.get("title"))]
    sampled[name] = random.sample(eligible, min(5, len(eligible)))

classifications = {}
for name, rows in sampled.items():
    classifications[name] = []
    print(f"\n  extraction : {name} ({len(rows)} textes)")
    for r in rows:
        text = r.get("selftext") or r.get("title", "")
        out = classify(text)
        classifications[name].append({"id": r.get("id"), **out})
        print(f"    {r.get('id')}: {out.get('cats', [])}")

total = sum(len(v) for v in classifications.values())
print(f"\nTotal classifications : {total}")



  extraction : Bumble_pre (5 textes)


    1cf8nor: ['INIT_NORM_W']


    1cfhy88: []


    1cfl0ul: []


    1cf6c8n: ['INIT_COST']


...

    1grc6gz: []


    1grhj9a: ['INIT_NORM_W', 'MATCH_DECAY']


    1grhu0e: ['INIT_NORM_W']


    1grgzuj: ['INIT_COST']


    1grgbh1: []

Total classifications : 20


In [6]:
# --- Distribution des categories par cellule ---
# Comptage : combien de textes portent >= 1 occurrence de la categorie.
# Multilabel : un texte peut porter plusieurs categories.

from collections import Counter
import pandas as pd

dist = {}
for name, rows in classifications.items():
    c = Counter()
    for r in rows:
        for cat in r.get("cats", []):
            c[cat] += 1
    dist[name] = dict(c)

df_dist = pd.DataFrame(dist, index=CATEGORIES).fillna(0).astype(int)
print("Distribution (count de textes portant la categorie) :")
print(df_dist)

delta_t = df_dist["Bumble_post"]  - df_dist["Bumble_pre"]
delta_c = df_dist["Hinge_post"]   - df_dist["Hinge_pre"]
delta_T = df_dist["Bumble_post"]  - df_dist["Hinge_post"]
delta_B = df_dist["Bumble_pre"]   - df_dist["Hinge_pre"]
did = delta_T - delta_B
print("\nDelta post-pre traite (Bumble)  :", delta_t.to_dict())
print("Delta post-pre controle (Hinge) :", delta_c.to_dict())
print("Delta DiD (traite - controle)   :", did.to_dict())


Distribution (count de textes portant la categorie) :
                    Bumble_pre  Bumble_post  Hinge_pre  Hinge_post
INIT_NORM_W                  1            1          0           2
INIT_NORM_ANY                0            0          0           0
INIT_COST                    1            2          0           1
BLAME                        1            1          0           0
MATCH_DECAY                  1            2          0           1
OPENING_MOVE_REACT           0            0          0           0

Delta post-pre traite (Bumble)  : {'INIT_NORM_W': 0, 'INIT_NORM_ANY': 0, 'INIT_COST': 1, 'BLAME': 0, 'MATCH_DECAY': 1, 'OPENING_MOVE_REACT': 0}
Delta post-pre controle (Hinge) : {'INIT_NORM_W': 2, 'INIT_NORM_ANY': 0, 'INIT_COST': 1, 'BLAME': 0, 'MATCH_DECAY': 1, 'OPENING_MOVE_REACT': 0}
Delta DiD (traite - controle)   : {'INIT_NORM_W': -2, 'INIT_NORM_ANY': 0, 'INIT_COST': 0, 'BLAME': 0, 'MATCH_DECAY': 0, 'OPENING_MOVE_REACT': 0}


In [7]:
# --- Mesure structurelle d_struct via l'instrument ---
# On construit 4 graphes AIF (un par cellule de mesure) en convertissant les
# classifications LLM en enonces dates (jour = rang, prop = hash(text),
# relations = multi-label). Comparaison pre/post sur chaque bras (DiD).

import hashlib

def text_to_prop(text):
    h = hashlib.md5(text.encode("utf-8")).hexdigest()[:6]
    return f"p{h}"

def classifications_to_enonces(rows, day_start):
    enonces = []
    for i, r in enumerate(rows):
        day = day_start + i
        text = r.get("verbatim") or r.get("id", "")
        prop = text_to_prop(text)
        cats = r.get("cats", [])
        rels = []
        for j, r2 in enumerate(rows):
            if i == j:
                continue
            cats2 = r2.get("cats", [])
            if len(cats) > len(cats2):
                rels.append(("CA", text_to_prop(
                    r2.get("verbatim") or r2.get("id", "")
                )))
        rels = tuple(sorted(set(rels)))
        enonces.append(Enonce(day, prop, "assertif", rels))
    return enonces

graphs = {}
day_bases = {"pre": 1, "post": 200}
for name in ("Bumble_pre", "Bumble_post", "Hinge_pre", "Hinge_post"):
    rows = classifications[name]
    period = "post" if "_post" in name else "pre"
    enonces = classifications_to_enonces(rows, day_bases[period])
    graphs[name] = build_aif_graph(enonces)
    print(f"  {name} : {len(graphs[name].i_nodes)} I-nodes, "
          f"{len(graphs[name].scheme_nodes)} schemes, "
          f"{len(graphs[name].excluded)} exclus")

d_bumble_pre_post = d_struct(graphs["Bumble_pre"], graphs["Bumble_post"])
d_hinge_pre_post  = d_struct(graphs["Hinge_pre"],  graphs["Hinge_post"])
delta_n = d_bumble_pre_post[0] - d_hinge_pre_post[0]
delta_a = d_bumble_pre_post[1] - d_hinge_pre_post[1]
print(f"\n  d_struct(Bumble)   : noeuds Δ={d_bumble_pre_post[0]:.3f}, "
      f"attaques Δ={d_bumble_pre_post[1]:.3f}")
print(f"  d_struct(controle) : noeuds Δ={d_hinge_pre_post[0]:.3f}, "
      f"attaques Δ={d_hinge_pre_post[1]:.3f}")
print(f"  DiD(noeuds)   = {delta_n:.3f}")
print(f"  DiD(attaques) = {delta_a:.3f}")


  Bumble_pre : 5 I-nodes, 8 schemes, 0 exclus
  Bumble_post : 5 I-nodes, 9 schemes, 0 exclus
  Hinge_pre : 0 I-nodes, 0 schemes, 1 exclus
  Hinge_post : 5 I-nodes, 8 schemes, 0 exclus

  d_struct(Bumble)   : noeuds Δ=1.000, attaques Δ=1.000
  d_struct(controle) : noeuds Δ=1.000, attaques Δ=1.000
  DiD(noeuds)   = 0.000
  DiD(attaques) = 0.000


In [8]:
# --- Plancher de bruit : split-half sur la fenetre pre de chaque bras ---
# Reprend le split_half de l'instrument, en re-calculant d_struct sur deux
# moities de la fenetre pre. Le DiD noeuds observe est-il > DiD plancher ?

import statistics as stats

def split_and_d_struct(enonces, n_reps=20, seed_base=0):
    deltas_n, deltas_a = [], []
    for k in range(n_reps):
        g1, g2 = split_half(enonces, seed=seed_base + k)
        try:
            af1 = build_aif_graph(g1)
            af2 = build_aif_graph(g2)
            d_n, d_a = d_struct(af1, af2)
            deltas_n.append(d_n)
            deltas_a.append(d_a)
        except Exception:
            continue
    if not deltas_n:
        return float("nan"), float("nan")
    return stats.median(deltas_n), stats.median(deltas_a)

floor_bumble = split_and_d_struct(
    classifications_to_enonces(classifications["Bumble_pre"], 1),
    n_reps=10,
)
floor_hinge = split_and_d_struct(
    classifications_to_enonces(classifications["Hinge_pre"], 1),
    n_reps=10,
)

floor_total_n = stats.median([floor_bumble[0], floor_hinge[0]])
floor_total_a = stats.median([floor_bumble[1], floor_hinge[1]])
print(f"Plancher noeuds   : Bumble pré = {floor_bumble[0]:.3f}, "
      f"Hinge pré = {floor_hinge[0]:.3f}, mediane = {floor_total_n:.3f}")
print(f"Plancher attaques : Bumble pré = {floor_bumble[1]:.3f}, "
      f"Hinge pré = {floor_hinge[1]:.3f}, mediane = {floor_total_a:.3f}")

above_floor_n = delta_n - floor_total_n
above_floor_a = delta_a - floor_total_a
print(f"\nDiD observe (noeuds)   Δ = {delta_n:.3f} ; au-dessus plancher : "
      f"{above_floor_n:.3f}")
print(f"DiD observe (attaques) Δ = {delta_a:.3f} ; au-dessus plancher : "
      f"{above_floor_a:.3f}")

VERDICT = "INCONCLUSIVE"
REASONS = []
if above_floor_n > 0.05:
    REASONS.append("DiD(noeuds) > plancher + 0.05")
if above_floor_a > 0.05:
    REASONS.append("DiD(attaques) > plancher + 0.05")

h1_signed = (
    delta_t.get("INIT_NORM_ANY", 0) > 0
    and delta_t.get("OPENING_MOVE_REACT", 0) > 0
    and delta_t.get("INIT_NORM_W", 0) <= 0
)
if h1_signed:
    REASONS.append("signes Δ traités cohérents H1 "
                   "(INIT_NORM_ANY↑, OPENING_MOVE_REACT↑, INIT_NORM_W↓)")

if above_floor_n > 0.05 and above_floor_a > 0.05 and h1_signed:
    VERDICT = "EXPLORATION"
elif above_floor_n <= 0.05 and above_floor_a <= 0.05:
    VERDICT = "INCONCLUSIVE"
else:
    VERDICT = "TRIANGULATION"

print(f"\nVERDICT : {VERDICT}")
for r in REASONS:
    print(f"  raison : {r}")


Plancher noeuds   : Bumble pré = 0.769, Hinge pré = 0.000, mediane = 0.385
Plancher attaques : Bumble pré = 1.000, Hinge pré = 0.000, mediane = 0.500

DiD observe (noeuds)   Δ = 0.000 ; au-dessus plancher : -0.385
DiD observe (attaques) Δ = 0.000 ; au-dessus plancher : -0.500

VERDICT : INCONCLUSIVE


## 4. Verdict et plafond de preuve

À la fin du notebook, la cellule précédente pose le verdict sur quatre observations conjointes :

1. **Plancher split-half** : médiane 10 répétitions sur la fenêtre *pré* de chaque bras.
2. **DiD(noeuds)** : `Δ_struct(Bumble) − Δ_struct(contrôle)`.
3. **DiD(attaques)** : idem sur les schemes.
4. **Signes H₁** : direction signée par catégorie (cohérence avec le prédict ).

**Verdict** : `EXPLORATION`, `TRIANGULATION` ou `INCONCLUSIVE`. **Jamais** de verdict plus haut qu'associatif (barreau 1) → barreau 2 atteignable seulement si tous les critères structurels sont satisfaits et que la taille d'échantillon (≥ 50 par cellule) a été tenue. **Barreau 3 récusé par construction** (#13309 §1).

## 5. Biais de sélection et plafond de barreau

r/Bumble sur-représente les utilisateurs investis et les événements saillants. Le **biais de saillance** pousse vers H₁ : la rupture elle-même fabrique des posts `OPENING_MOVE_REACT`. Le bras contrôle (r/hingeapp) n'absorbe pas ce biais car il ne porte **aucune rupture** d'initiation sur la fenêtre (Hinge conserve le modèle traditionnel « n'importe qui peut écrire en premier »). Le contrôle DiD reste imparfait : il estomac le biais de plateforme (Reddit) et de fenêtre, pas le biais de sujet.

**Plafond revendiqué d'avance** : barreau associatif. **Quasi-expérimental** inatteignable avec ce corpus (manquent : utilisateurs silencieux, résiliés, hors-app, non-anglophones).

## 6. Limites assumées et extensions futures

- **Sous-échantillon de 5 textes / cellule** : suffisant pour démontrer l'instrumentation et capturer le verdict, **insuffisant** pour une robustesse statistique. L'extraction à 50 textes / cellule est marquée comme extension §11 du corps de l'EPIC.
- **Pas de corpus brut committé** : seuls les agrégats et 5 verbatims bornés par cellule sont commit-éligibles (politique dataset CLAUDE.md).
- **Modèle local Ollama** : ré-exécution déterministe (température 0, prompt épinglé), mais la version `qwen2.5:7b-instruct-q4_K_M` est figée par l'épinglage du SHA dans le notebook ; toute mise à jour du modèle = PR dédiée.


## 7. Conclusion

Premier cas de l'Observatoire livré. **Instrument Dated_Graphs rebranché, pas réécrit**. **Rupture datée, fenêtres symétriques, bras contrôle intact**. Verdict borné par construction (barreau 1 max).

Voir aussi :

- [`Argument_Analysis_Dated_Graphs.ipynb`](Argument_Analysis_Dated_Graphs.ipynb) (#13310, l'instrument)
- Issue #13303 (EPIC Observatoire)
- Issue #13309 (protocole de preuve)
- Commentaire d'inscription #16410 (pré-inscription datée, axes signés)
